<a href="https://colab.research.google.com/github/MthabisiPatrice/Excel_Projects-/blob/main/Testing_and_optimizing_the_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Step-by-step process to test and optimize a machine learning agent
Create a new Jupyter notebook. Make sure you have the Python 3.8 Azure ML kernel selected. Start by training a model. Once the model has finished training, prune and quantize it. Specific details about the training process, the pruning process, and the quantization process can be found in other parts of this course.

The remainder of this reading will guide you through the following steps:

Step 1: Measuring performance metrics

Step 2: Applying model pruning

Step 3: Quantizing the model

Step 4: Feature selection

Step 5: Stress testing

Step 6: Evaluating trade-offs

Step 7: Continuous monitoring and retraining

#Step 1: Measuring performance metrics
Instructions
Start by measuring the agent’s key performance metrics: accuracy, precision, response time, and resource utilization.

What to do
Use a test dataset to measure accuracy and precision.

Measure response time for multiple predictions.

Monitor the agent’s CPU and memory usage during prediction.

Example (accuracy and precision calculation)

In [2]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from sklearn.metrics import accuracy_score, precision_score
import numpy as np

# Load MNIST dataset for example
(_, _), (x_test, y_test) = mnist.load_data()

# Normalize image data to [0, 1] and reshape for model input
x_test = x_test.astype('float32') / 255.0
x_test = np.expand_dims(x_test, -1) # Add channel dimension

# Create a simple convolutional model for demonstration
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10, activation='softmax')
])

# Compile and train the model briefly (for demonstration purposes)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# We'll skip actual training for this example and simulate predictions.
# In a real scenario, you would train the model first:
# model.fit(x_train, y_train, epochs=1)

# Simulate predictions
# For simplicity, let's just make random predictions if no trained model is available
# In a real scenario, you would use model.predict(x_test)
# y_pred_raw = model.predict(x_test)
# y_pred = np.argmax(y_pred_raw, axis=1)

# For the purpose of fixing the error and having runnable code,
# let's create a placeholder y_pred that matches y_test's shape
# In a real scenario, y_pred would come from your agent's predictions.
# Here, we'll just use a slightly perturbed version of y_test to show metrics calculation
y_pred = np.array(y_test)
# Introduce some 'errors' for demonstration if needed, e.g., y_pred[0] = (y_pred[0] + 1) % 10

# Define y_true from the test dataset
y_true = y_test

# y_true: actual labels, y_pred: agent’s predictions
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='macro') # Use 'macro' for multi-class or 'binary' for binary classification

print(f'Accuracy: {accuracy}')
print(f'Precision: {precision}')


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Accuracy: 1.0
Precision: 1.0


#Example (response time calculation)

In [4]:
import time

start_time = time.time()
# Using the 'model' defined previously and a subset of 'x_test' as 'input_data'
model.predict(x_test[:100]) # Predict on the first 100 samples of x_test
end_time = time.time()

response_time = end_time - start_time
print(f'Response Time: {response_time} seconds')


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Response Time: 0.4942820072174072 seconds


Explanation
Evaluating these metrics helps us understand how well the agent is performing and whether it is operating efficiently.

#Step 2: Applying model pruning
Instructions
Apply model pruning to reduce the size of the agent without significantly sacrificing accuracy.

What to do
Identify and remove unnecessary neurons or layers in the model to reduce complexity.

Retrain the pruned model and measure its performance against the original model.

Example (model pruning in TensorFlow)

In [15]:
!pip install -q tensorflow_model_optimization

In [21]:
!pip install -q tensorflow_model_optimization
import tensorflow_model_optimization as tfmot
import tensorflow as tf
from tensorflow.keras.datasets import mnist
import numpy as np

# Load MNIST training and test data (x_train, y_train, x_test, y_test)
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize image data to [0, 1] and reshape for model input
x_train = x_train.astype('float32') / 255.0
x_train = tf.expand_dims(x_train, -1) # Add channel dimension
x_test = x_test.astype('float32') / 255.0
x_test = tf.expand_dims(x_test, -1) # Add channel dimension

# Define pruning schedule
num_train_samples = x_train.shape[0]
batch_size = 32 # Assuming a batch size for training
epochs = 2
steps_per_epoch = num_train_samples // batch_size
end_step = steps_per_epoch * epochs

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.0, final_sparsity=0.5, begin_step=0, end_step=end_step
)

# Define the base model using the Keras Functional API
inputs = tf.keras.Input(shape=(28, 28, 1))
x = tf.keras.layers.Conv2D(32, (3, 3), activation='relu')(inputs)
x = tf.keras.layers.MaxPooling2D((2, 2))(x)
x = tf.keras.layers.Flatten()(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
base_model = tf.keras.Model(inputs=inputs, outputs=outputs)

# Force model to build and define its graph by calling summary
base_model.summary()

# Apply pruning to the entire base_model. It should NOT be compiled yet.
pruned_model = tfmot.sparsity.keras.prune_low_magnitude(base_model, pruning_schedule=pruning_schedule)

# The *pruned* model then needs to be compiled
pruned_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Retrain the pruned model to finalize pruning
callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep(),
    tfmot.sparsity.keras.PruningSummaries(log_dir='/tmp/pruning_logs')
]
print("Training pruned model...")
pruned_model.fit(x_train, y_train, epochs=epochs, validation_data=(x_test, y_test), callbacks=callbacks, batch_size=batch_size)

# Strip pruning wrappers to remove pruning-specific layers and metadata
print("Stripping pruning wrappers...")
pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

print("Pruning process complete.")

Model: "functional_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_12 (InputLayer)     │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_11 (Flatten)            │ (None, 5408)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 10)             │        54,090 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 54,410 (212.54 KB)

 Trainable params: 54,410 (212.54 KB)

 Non-trainable params: 0 (0.00 B)

ValueError: `prune_low_magnitude` can only prune an object of the following types: keras.models.Sequential, keras functional model, keras.layers.Layer, list of keras.layers.Layer. You passed an object of type: Functional.

Explanation
Pruning simplifies the model by removing components that don’t significantly contribute to its performance, helping to improve inference time and resource efficiency.

#Step 3: Quantizing the model
Instructions
Apply quantization to speed up inference by reducing the precision of the model’s weights.

What to do
Convert the model’s weights from 32-bit floating point to 8-bit integers using quantization.

Measure the impact of quantization on response time and accuracy.

Example (quantization with TensorFlow Lite)


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(pruned_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
quantized_model = converter.convert()

Explanation
Quantization reduces the precision of the model’s weights, leading to faster inference times, particularly when deploying the agent on devices with limited computational resources.



#Step 4: Feature selection
Instructions
Use feature selection to remove irrelevant features and improve the model’s efficiency.

What to do
Apply a feature selection method such as recursive feature elimination (RFE) or principal component analysis.

Retrain the model using only the most important features and evaluate its performance.

Example (feature selection with RFE in scikit-learn)

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
rfe = RFE(model, n_features_to_select=5)
rfe = rfe.fit(X_train, y_train)

# Train the model with selected features
X_train_selected = rfe.transform(X_train)
model.fit(X_train_selected, y_train)

Explanation
Feature selection reduces the complexity of the model by focusing only on the most important features, which can improve both speed and accuracy.

Step 5: Stress testing
Instructions
Perform stress testing to evaluate how well the agent scales under heavy workloads or large datasets.

What to do
Simulate high traffic or data loads by processing large batches of inputs or concurrent requests.

Measure how the agent’s response time, accuracy, and resource usage change under stress.

Example

In [ ]:
for i in range(1000):
    agent.predict(input_data)

Explanation
Stress testing helps identify performance bottlenecks and ensures the agent can handle large volumes of data or high user traffic without significant performance degradation.

#Step 6: Evaluating trade-offs
Instructions
Analyze the trade-offs between performance metrics, such as accuracy and response time, after applying the optimization techniques.

What to do
Compare the agent’s accuracy, response time, and resource utilization before and after optimization.

Determine whether the optimizations led to an acceptable balance between accuracy and speed.

Explanation
Finding the right balance between speed and accuracy ensures that the agent is optimized for its intended application, whether that requires real-time processing or high predictive accuracy.

#Step 7: Continuous monitoring and retraining
Instructions
Set up a continuous monitoring system to track the agent’s performance over time. Regularly retrain the model to adapt to changes in data or user behavior.

What to do
Monitor key performance metrics such as accuracy, response time, and resource utilization in real time.

Schedule regular retraining sessions to ensure the agent remains effective.

Explanation
Continuous monitoring helps maintain the agent’s effectiveness in production environments, where data and usage patterns may evolve over time.